# 07 — BioASQ + PubMed Retrieval Dataset

Builds `data/processed/bioasq_pubmed_retrieval.csv` — a retrieval-ready dataset that pairs every BioASQ question with:
- Its **ground-truth answer** (exact + ideal)
- The **supporting PubMed snippets/abstracts** used as retrieval context
- A structured **Plan of Action (POA)** column derived from the ideal answer

### Sources
| Source | Role |
|---|---|
| `data/raw/BioASQ/7B_golden.json` | Gold-standard test questions with rich metadata |
| `data/raw/BioASQ/BioASQ-train-factoid-{4-7}b.json` | Factoid training questions (SQuAD-style) |
| `data/raw/BioASQ/BioASQ-train-yesno-7b.json` | Yes/No training questions |
| `data/raw/BioASQ/{4-7}B{1-5}_golden.json` | All golden test sets |
| `data/raw/pubmed_abstracts.csv` | PubMed abstract corpus (8 topics, wide format) |

### Output schema
| Column | Description |
|---|---|
| `question_id` | Unique BioASQ question ID |
| `question` | Question text |
| `answer_type` | `factoid` / `list` / `yesno` / `summary` |
| `ground_truth_answer` | Exact answer (string) |
| `ideal_answer` | Natural-language ideal answer |
| `snippets_text` | Pipe-separated supporting passages |
| `snippet_pmids` | Pipe-separated PMIDs for those passages |
| `all_doc_pmids` | All document PMIDs referenced |
| `pubmed_title` | Matched PubMed title (if available) |
| `pubmed_abstract` | Matched PubMed abstract (if available) |
| `pubmed_topic` | Topic category from pubmed_abstracts.csv |
| `plan_of_action` | Structured POA derived from ideal answer |
| `source_file` | Which BioASQ file this came from |

In [ ]:
import json
import ast
import re
import os
import glob
import pandas as pd
from pathlib import Path

ROOT = Path("..")
RAW_BIOASQ = ROOT / "data" / "raw" / "BioASQ"
RAW_PUBMED = ROOT / "data" / "raw" / "pubmed_abstracts.csv"
OUT_DIR    = ROOT / "data" / "processed"
OUT_PATH   = OUT_DIR / "bioasq_pubmed_retrieval.csv"

print("BioASQ dir:", RAW_BIOASQ.resolve())
print("PubMed CSV:", RAW_PUBMED.resolve())
print("Output   :", OUT_PATH.resolve())

## 1. Helper utilities

In [ ]:
def pmid_from_url(url: str) -> str:
    """Extract numeric PMID from a PubMed URL or return the raw string."""
    if not isinstance(url, str):
        return ""
    m = re.search(r"pubmed/(\d+)", url)
    return m.group(1) if m else url.strip()


def flatten_exact_answer(exact_answer, answer_type: str) -> str:
    """
    Normalise exact_answer across BioASQ question types:
      factoid / list  → list-of-lists  e.g. [["ans1","syn"], ["ans2"]]
      yesno           → plain string   "yes" / "no"
      summary         → absent / None
    Returns a readable string.
    """
    if exact_answer is None:
        return ""
    if isinstance(exact_answer, str):
        return exact_answer.strip()
    if isinstance(exact_answer, list):
        # Flatten inner lists with " / " and join outer with " | "
        parts = []
        for item in exact_answer:
            if isinstance(item, list):
                parts.append(" / ".join(str(x) for x in item))
            else:
                parts.append(str(item))
        return " | ".join(parts)
    return str(exact_answer)


def first_ideal_answer(ideal_answer) -> str:
    """Return the first ideal answer string."""
    if not ideal_answer:
        return ""
    if isinstance(ideal_answer, list):
        return ideal_answer[0].strip()
    return str(ideal_answer).strip()


def derive_plan_of_action(ideal_answer: str, answer_type: str, exact_answer: str) -> str:
    """
    Build a structured Plan-of-Action (POA) string from the available answer fields.

    POA template:
      [Step 1] Identify the clinical/research question.
      [Step 2] Retrieve supporting evidence from PubMed snippets.
      [Step 3] Ground-truth answer: <exact_answer>
      [Step 4] Synthesise: <condensed ideal_answer>
      [Step 5] Recommend: <action / decision based on answer type>
    """
    if not ideal_answer and not exact_answer:
        return ""

    # Condense ideal answer to ≤200 chars for POA
    summary = ideal_answer[:200].rstrip() + ("..." if len(ideal_answer) > 200 else "")

    if answer_type == "yesno":
        recommendation = (
            f"Clinical recommendation: The answer is '{exact_answer.upper()}'. "
            "Proceed or withhold intervention accordingly."
        )
    elif answer_type == "factoid":
        recommendation = (
            f"Clinical recommendation: The specific entity/value is '{exact_answer}'. "
            "Use this as the target in downstream queries or treatment decisions."
        )
    elif answer_type == "list":
        items = exact_answer.split(" | ") if exact_answer else []
        recommendation = (
            f"Clinical recommendation: Consider each of the following ({len(items)} items): "
            + "; ".join(items[:5])
            + (". (truncated)" if len(items) > 5 else ".")
        )
    else:  # summary
        recommendation = (
            "Clinical recommendation: Review the synthesised evidence above "
            "and formulate a management plan with the treating clinician."
        )

    poa = (
        "[Step 1] Identify the clinical/research question.\n"
        "[Step 2] Retrieve supporting evidence from PubMed snippets.\n"
        f"[Step 3] Ground-truth answer: {exact_answer}\n"
        f"[Step 4] Synthesise: {summary}\n"
        f"[Step 5] {recommendation}"
    )
    return poa


print("Helpers defined.")

## 2. Parse BioASQ *golden* files (native format)

These are the richest files: `7B_golden.json`, `6B{1-5}_golden.json`, etc.  
Each question carries: `body`, `type`, `documents`, `snippets`, `ideal_answer`, `exact_answer`.

In [ ]:
def parse_golden_file(path: Path) -> list[dict]:
    """
    Parse a native BioASQ golden JSON file.
    Returns a list of flat record dicts.
    """
    with open(path, encoding="utf-8") as f:
        data = json.load(f)

    records = []
    questions = data.get("questions", [])

    for q in questions:
        qid          = q.get("id", "")
        body         = q.get("body", "")
        qtype        = q.get("type", "unknown")
        documents    = q.get("documents", [])
        snippets     = q.get("snippets", [])
        ideal_answer = first_ideal_answer(q.get("ideal_answer", ""))
        exact        = flatten_exact_answer(q.get("exact_answer", None), qtype)

        # Flatten snippets
        snippet_texts = [s.get("text", "").strip() for s in snippets]
        snippet_pmids = [pmid_from_url(s.get("document", "")) for s in snippets]

        # All referenced PMIDs
        all_pmids = [pmid_from_url(d) for d in documents]

        records.append({
            "question_id"       : qid,
            "question"          : body,
            "answer_type"       : qtype,
            "ground_truth_answer": exact,
            "ideal_answer"      : ideal_answer,
            "snippets_text"     : " ||| ".join(snippet_texts),
            "snippet_pmids"     : "|".join(snippet_pmids),
            "all_doc_pmids"     : "|".join(all_pmids),
            "source_file"       : path.name,
        })

    return records


# Collect all golden files
golden_files = sorted(RAW_BIOASQ.glob("*_golden.json"))
print(f"Found {len(golden_files)} golden files:")
for f in golden_files:
    print(" ", f.name)

In [ ]:
all_golden_records = []
for gf in golden_files:
    recs = parse_golden_file(gf)
    print(f"  {gf.name:45s} → {len(recs):4d} questions")
    all_golden_records.extend(recs)

df_golden = pd.DataFrame(all_golden_records)
print(f"\nTotal golden records : {len(df_golden):,}")
print(df_golden["answer_type"].value_counts())

## 3. Parse BioASQ *train* files (SQuAD-style)

These files use a different schema — SQuAD-style with `data → paragraphs → context / qas`.  
Multiple paragraphs exist per question (one per supporting passage).  
We de-duplicate by `question_id` and aggregate all contexts.

In [ ]:
def parse_squad_file(path: Path, answer_type: str) -> list[dict]:
    """
    Parse a SQuAD-style BioASQ train JSON file.
    `answer_type` is 'factoid' or 'yesno' (inferred from filename).
    Returns one record per unique question (contexts aggregated).
    """
    with open(path, encoding="utf-8") as f:
        data = json.load(f)

    # Accumulate per question_id
    q_map: dict[str, dict] = {}  # base_id → record

    for group in data.get("data", []):
        for para in group.get("paragraphs", []):
            context = para.get("context", "").strip()
            for qa in para.get("qas", []):
                full_id  = qa.get("id", "")
                # Strip the trailing _NNN paragraph index to get the base question id
                base_id  = re.sub(r"_\d{3}$", "", full_id)
                question = qa.get("question", "").strip()

                # Extract answer depending on type
                raw_answers = qa.get("answers", None)
                if answer_type == "yesno":
                    # answers is a plain string "yes" / "no"
                    ans_text = raw_answers if isinstance(raw_answers, str) else ""
                else:
                    # answers is a list of {text, answer_start}
                    if isinstance(raw_answers, list) and raw_answers:
                        ans_text = raw_answers[0].get("text", "")
                    else:
                        ans_text = ""

                if base_id not in q_map:
                    q_map[base_id] = {
                        "question_id"        : base_id,
                        "question"           : question,
                        "answer_type"        : answer_type,
                        "ground_truth_answer": ans_text,
                        "ideal_answer"       : ans_text,   # no long answer in train files
                        "snippets_text"      : context,
                        "snippet_pmids"      : "",         # not available in SQuAD format
                        "all_doc_pmids"      : "",
                        "source_file"        : path.name,
                        "_contexts"          : [context],
                    }
                else:
                    # Append additional context paragraphs
                    q_map[base_id]["_contexts"].append(context)

    # Finalise: join all contexts
    records = []
    for rec in q_map.values():
        all_ctx = list(dict.fromkeys(rec.pop("_contexts")))  # deduplicate preserving order
        rec["snippets_text"] = " ||| ".join(all_ctx)
        records.append(rec)

    return records


# Discover all train files
factoid_train_files = sorted(RAW_BIOASQ.glob("BioASQ-train-factoid-*.json"))
yesno_train_files   = sorted(RAW_BIOASQ.glob("BioASQ-train-yesno-*.json"))
print("Factoid train files:", [f.name for f in factoid_train_files])
print("Yesno  train files :", [f.name for f in yesno_train_files])

In [ ]:
all_train_records = []

for tf in factoid_train_files:
    recs = parse_squad_file(tf, answer_type="factoid")
    print(f"  {tf.name:45s} → {len(recs):4d} questions")
    all_train_records.extend(recs)

for tf in yesno_train_files:
    recs = parse_squad_file(tf, answer_type="yesno")
    print(f"  {tf.name:45s} → {len(recs):4d} questions")
    all_train_records.extend(recs)

df_train = pd.DataFrame(all_train_records)
print(f"\nTotal train records  : {len(df_train):,}")
print(df_train["answer_type"].value_counts())

## 4. Merge golden + train, deduplicate

In [ ]:
df_bioasq = pd.concat([df_golden, df_train], ignore_index=True)

print(f"Combined (before dedup): {len(df_bioasq):,}")

# Drop exact duplicates on question_id — keep golden over train (golden appears first)
df_bioasq = df_bioasq.drop_duplicates(subset="question_id", keep="first")

print(f"After dedup by question_id: {len(df_bioasq):,}")
print("Answer type breakdown:")
print(df_bioasq["answer_type"].value_counts())

## 5. Parse PubMed abstracts CSV — wide → long

The CSV is wide-format with 8 topic columns. Each cell is a Python tuple string `([abstract], 'title')`.  
We melt it to one row per document and map PMIDs from the corresponding URL columns.

In [ ]:
TOPIC_COLS = [
    "deep_learning", "covid_19", "human_connectome", "virtual_reality",
    "brain_machine_interfaces", "electroactive_polymers", "pedot_electrodes",
    "neuroprosthetics",
]
LINK_COLS = [f"{t}_links" for t in TOPIC_COLS]


def safe_literal_eval(val):
    """Parse the cell value as a Python literal; return ([], '') on failure."""
    if not isinstance(val, str) or not val.strip():
        return ([], "")
    try:
        result = ast.literal_eval(val.strip())
        # Expected: ([abstract_text, ...], title_string)
        if isinstance(result, tuple) and len(result) == 2:
            abstracts, title = result
            abstract = abstracts[0] if isinstance(abstracts, list) and abstracts else ""
            return (abstract, str(title))
    except Exception:
        pass
    return ("", str(val)[:120])


df_pub_raw = pd.read_csv(RAW_PUBMED)
print(f"PubMed CSV shape: {df_pub_raw.shape}")
print("Columns:", list(df_pub_raw.columns))

In [ ]:
pubmed_records = []

for _, row in df_pub_raw.iterrows():
    for topic, link_col in zip(TOPIC_COLS, LINK_COLS):
        cell     = row.get(topic, "")
        url      = str(row.get(link_col, "")).strip()
        pmid     = pmid_from_url(url)

        abstract, title = safe_literal_eval(cell)

        if not pmid and not abstract and not title:
            continue  # skip fully empty cells

        pubmed_records.append({
            "pmid"           : pmid,
            "pubmed_title"   : title.strip(),
            "pubmed_abstract": abstract.strip(),
            "pubmed_topic"   : topic,
            "pubmed_url"     : url,
        })

df_pubmed = pd.DataFrame(pubmed_records)

# Deduplicate: keep one row per PMID (first occurrence wins)
df_pubmed = df_pubmed[df_pubmed["pmid"].str.len() > 0]
df_pubmed_dedup = df_pubmed.drop_duplicates(subset="pmid", keep="first")

print(f"PubMed long-format rows  : {len(df_pubmed):,}")
print(f"After PMID dedup         : {len(df_pubmed_dedup):,}")
print("Topics:", df_pubmed["pubmed_topic"].value_counts().to_dict())

## 6. Join BioASQ → PubMed abstracts via PMID

Strategy:
1. Explode `snippet_pmids` (pipe-separated) to find the first available PMID per question.
2. Left-join to PubMed deduplicated table on PMID.
3. For questions with multiple PMIDs, keep the first successful match.

In [ ]:
def first_pmid(pmid_string: str) -> str:
    """Return the first PMID from a pipe-separated string."""
    if not isinstance(pmid_string, str):
        return ""
    parts = [p.strip() for p in pmid_string.split("|") if p.strip()]
    return parts[0] if parts else ""


# Extract first snippet PMID and first doc PMID for joining
df_bioasq["_join_pmid"] = df_bioasq["snippet_pmids"].apply(first_pmid)
# Fall back to first doc PMID if no snippet PMIDs
mask_empty = df_bioasq["_join_pmid"] == ""
df_bioasq.loc[mask_empty, "_join_pmid"] = df_bioasq.loc[mask_empty, "all_doc_pmids"].apply(first_pmid)

print(f"Questions with a join PMID : {(df_bioasq['_join_pmid'] != '').sum():,}")
print(f"Questions without any PMID : {(df_bioasq['_join_pmid'] == '').sum():,}")

In [ ]:
# Merge
df_merged = df_bioasq.merge(
    df_pubmed_dedup[["pmid", "pubmed_title", "pubmed_abstract", "pubmed_topic"]],
    left_on="_join_pmid",
    right_on="pmid",
    how="left",
).drop(columns=["_join_pmid", "pmid"])

# Fill NaN text columns with empty string
for col in ["pubmed_title", "pubmed_abstract", "pubmed_topic"]:
    df_merged[col] = df_merged[col].fillna("")

matched = (df_merged["pubmed_abstract"] != "").sum()
print(f"Rows with matched PubMed abstract : {matched:,} / {len(df_merged):,} ({matched/len(df_merged)*100:.1f}%)")

## 7. Generate Plan of Action (POA) column

In [ ]:
df_merged["plan_of_action"] = df_merged.apply(
    lambda row: derive_plan_of_action(
        ideal_answer=row["ideal_answer"],
        answer_type=row["answer_type"],
        exact_answer=row["ground_truth_answer"],
    ),
    axis=1,
)

print(f"POA generated for {(df_merged['plan_of_action'] != '').sum():,} rows.")
print("\nSample POA (row 0):")
print(df_merged["plan_of_action"].iloc[0])

## 8. Final column selection & quality checks

In [ ]:
FINAL_COLS = [
    "question_id",
    "question",
    "answer_type",
    "ground_truth_answer",
    "ideal_answer",
    "snippets_text",
    "snippet_pmids",
    "all_doc_pmids",
    "pubmed_title",
    "pubmed_abstract",
    "pubmed_topic",
    "plan_of_action",
    "source_file",
]

df_final = df_merged[FINAL_COLS].copy()

# Drop rows with no question text (malformed entries)
before = len(df_final)
df_final = df_final[df_final["question"].str.strip() != ""]
print(f"Removed {before - len(df_final)} empty-question rows.")

# Reset index
df_final = df_final.reset_index(drop=True)

print(f"\nFinal dataset shape: {df_final.shape}")
print("\nColumn null counts:")
print(df_final.isnull().sum())

In [ ]:
# Summary statistics
print("=" * 60)
print("DATASET SUMMARY")
print("=" * 60)
print(f"Total questions          : {len(df_final):,}")
print(f"Unique question IDs      : {df_final['question_id'].nunique():,}")
print()
print("By answer type:")
print(df_final["answer_type"].value_counts().to_string())
print()
print("By source file (top 10):")
print(df_final["source_file"].value_counts().head(10).to_string())
print()
print(f"With ground_truth_answer : {(df_final['ground_truth_answer'] != '').sum():,}")
print(f"With ideal_answer        : {(df_final['ideal_answer'] != '').sum():,}")
print(f"With snippets            : {(df_final['snippets_text'] != '').sum():,}")
print(f"With PubMed abstract     : {(df_final['pubmed_abstract'] != '').sum():,}")
print(f"With POA                 : {(df_final['plan_of_action'] != '').sum():,}")

## 9. Save dataset

In [ ]:
OUT_DIR.mkdir(parents=True, exist_ok=True)
df_final.to_csv(OUT_PATH, index=False, encoding="utf-8")
size_mb = OUT_PATH.stat().st_size / 1024 / 1024
print(f"Saved → {OUT_PATH}")
print(f"File size: {size_mb:.2f} MB")
print(f"Rows: {len(df_final):,}  |  Columns: {len(df_final.columns)}")

## 10. Inspect a few sample rows

In [ ]:
pd.set_option("display.max_colwidth", 120)

print("── Factoid sample ──")
factoid_sample = df_final[df_final["answer_type"] == "factoid"].head(2)
display(factoid_sample[["question", "ground_truth_answer", "ideal_answer", "pubmed_topic"]].T)

print("\n── Yes/No sample ──")
yesno_sample = df_final[df_final["answer_type"] == "yesno"].head(2)
display(yesno_sample[["question", "ground_truth_answer", "ideal_answer"]].T)

print("\n── List sample ──")
list_sample = df_final[df_final["answer_type"] == "list"].head(2)
display(list_sample[["question", "ground_truth_answer"]].T)

In [ ]:
# Show a complete record with POA
print("── Complete record (with POA) ──")
sample = df_final[df_final["plan_of_action"] != ""].iloc[0]
for col in FINAL_COLS:
    val = sample[col]
    # Truncate long fields for display
    if isinstance(val, str) and len(val) > 300:
        val = val[:300] + "..."
    print(f"\n[{col}]\n{val}")

---
## Dataset complete

`data/processed/bioasq_pubmed_retrieval.csv` is ready for:
- **Ground-truth retrieval evaluation** — use `snippets_text` / `pubmed_abstract` as the retrieved context and `ground_truth_answer` / `ideal_answer` as the reference.
- **Plan-of-Action recommendation** — the `plan_of_action` column provides a structured 5-step POA for each question.
- **RAG pipeline input** — index `pubmed_abstract` + `snippets_text` in a vector store; use `question` as the query; compare retrieved answers against `ground_truth_answer`.